### CodeSmellEval Analysis (Dataset)

In [1]:
def default_params(): 
    return {
        'dataset' : '/workspaces/CodeSmells/semeru-datasets/pylint/distinct_repo_code_dataset_not_errors.json',
    }
params = default_params()

### Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns; sns.set_theme()
from collections import Counter
import plotly.express as px

### Load Dataset

In [3]:
raw_dataset = pd.read_json(params['dataset'])

### Dataset Curation

In [4]:
# Filter rows where pylint_analysis is a non-empty array
s_dataset = raw_dataset[raw_dataset["pylint_analysis"].apply(lambda x: isinstance(x, list) and len(x) > 0)]

In [5]:
s_dataset = s_dataset.explode('pylint_analysis').reset_index(drop=True)
# Normalize the JSON objects in the 'pylint_analysis' column
json_df = pd.json_normalize(s_dataset['pylint_analysis'])
# Drop the original 'pylint_analysis' column and join the new DataFrame
s_dataset = s_dataset.drop(columns=['pylint_analysis']).join(json_df)

In [6]:
# Renaming
s_dataset.rename(columns={'msg_id': 's_msg_id', 'line' : 's_line', 'column': 's_column', 'end_line': 's_end_line', 'end_column': 's_end_column', 'code_smell': 's_code'}, inplace=True)

In [7]:
# Map categories
categories_mapping = {
    'E' : 'Error', 
    'W' : 'Warning', 
    'F' : 'Fatal',
    'C' : 'Convention',
    'R' : 'Refactor',
    'I' : 'Information'
}
s_dataset['category'] = s_dataset['s_msg_id'].map(lambda x: categories_mapping[x[0]])

### Fix pylint script results

In [8]:
def extract_substring(code_string, start_line, start_column, end_line, end_column):
    # Split the string into individual lines.
    lines = code_string.splitlines()
    # Validate the provided indices.
    if start_line < 0 or start_line >= len(lines):
        raise IndexError("start_line is out of range.")
    if end_line < 0 or end_line >= len(lines):
        raise IndexError("end_line is out of range.")
    # Case when the substring is within a single line.
    if start_line == end_line:
        return lines[start_line][start_column:end_column]
    # Extract parts from multiple lines.
    # 1. Extract from the start line starting at start_column.
    extracted_lines = [lines[start_line][start_column:]]
    # 2. Add all the lines between the start and end lines (if any).
    for line in lines[start_line + 1 : end_line]:
        extracted_lines.append(line)
    # 3. Extract from the end line up to end_column.
    extracted_lines.append(lines[end_line][:end_column])
    # Join the parts with newline characters.
    return "\n".join(extracted_lines)

In [9]:
def compute_s_end_column(row):
    """Compute the s_end_column value for a given row, ensuring index safety."""
    if pd.notna(row['s_end_column']):
        return row['s_end_column']
    code_lines = row['code'].splitlines()
    
    # Validate that s_end_line is within the bounds of code_lines
    if row['s_end_line'] < 0 or row['s_end_line'] >= len(code_lines):
       raise IndexError("s_end_line is out of range.")
    
    return len(code_lines[row['s_end_line']])


In [10]:
### remove Errors
s_dataset = s_dataset[s_dataset['category']!='Error']

In [11]:
## Fixing from pylint scripts from galeras
### decrease values of lines by 1 
s_dataset['s_line'] = s_dataset['s_line'].apply(lambda x: x-1)
s_dataset['s_end_line'] = s_dataset['s_end_line'].apply(lambda x: x-1)
### fix type in index columns 
s_dataset['s_line'] = s_dataset['s_line'].astype('Int64')
s_dataset['s_column'] = s_dataset['s_column'].astype('Int64')
s_dataset['s_end_line'] = s_dataset['s_end_line'].astype('Int64')
s_dataset['s_end_column'] = s_dataset['s_end_column'].astype('Int64')

In [12]:
### if end_line and end_column is nan, compute them
s_dataset['s_end_line'] = s_dataset['s_end_line'].fillna(s_dataset['s_line'])
s_dataset['s_end_column'] = s_dataset.apply(compute_s_end_column, axis=1)
### replace s_code
s_dataset['s_code'] = s_dataset.apply(lambda row: extract_substring(row['code'], row['s_line'], row['s_column'], row['s_end_line'], row['s_end_column']), axis=1)

In [13]:
s_dataset.columns

Index(['id', 'commit_id', 'repo', 'path', 'file_name', 'fun_name',
       'commit_message', 'code', 'url', 'language', 'ast_errors',
       'n_ast_errors', 'ast_levels', 'n_whitespaces', 'n_words', 'vocab_size',
       'complexity', 'nloc', 'token_counts', 'n_ast_nodes', 'n_identifiers',
       's_msg_id', 's_line', 's_column', 's_end_line', 's_end_column',
       's_code', 'category'],
      dtype='object')

In [31]:
s_dataset[s_dataset['s_msg_id'] == 'W0106'].loc[4149840]['code']

'scheduler.step_plms(self.dummy_sample, 1, self.dummy_sample).prev_sample'

### Plot message ids

In [21]:
# extract message_ids
#msg_ids = np.concatenate(raw_dataset["pylint_analysis"].map(lambda code_smells: [code_smell['msg_id'] for code_smell in code_smells]).to_numpy())
msg_ids = s_dataset['s_msg_id'].to_numpy()
# Count frequenciess
msg_counts = Counter(msg_ids)
# Dataframe from counts 
df_msg_ids = pd.DataFrame(msg_counts.items(), columns=['s_msg_id', 'frequency'])
df_msg_ids['category'] = df_msg_ids['s_msg_id'].map(lambda x: categories_mapping[x[0]])

In [22]:
### Filter, with at least 1000 samples
df_msg_ids = df_msg_ids[df_msg_ids['frequency']>=500]

In [23]:
# Sort the dataframe by frequency in descending order
df_msg_ids = df_msg_ids.sort_values(by='frequency', ascending=False)
fig = px.treemap(df_msg_ids, path=['category', 's_msg_id'], custom_data=['frequency'], title="Message Ids Frequency Counts", )
fig.update_traces(texttemplate='%{label}<br>Freq: %{customdata[0]}', textfont_size=14)
fig.show()